# MorningNote AI — Demo Notebook

**Course:** Generative AI for Finance  
**Project:** MorningNote AI: An Agentic Financial Briefing Generator for Equity Watchlists

This notebook runs the full workflow from sector selection to model comparison.
Run all cells from top to bottom.

---

## Contents
1. [Setup & Imports](#section-1)
2. [User Selects Sector](#section-2)
3. [Build Watchlist](#section-3)
4. [Fetch Market Data](#section-4)
5. [Load Company News](#section-5)
6. [Build Prompt](#section-6)
7. [Generate Morning Notes (Two Models)](#section-7)
8. [Baseline — No AI](#section-8)
9. [Compare Model Outputs](#section-9)
10. [Fact-Check Claims](#section-10)
11. [Conclusion and Limitations](#section-11)

---
<a id='section-1'></a>
## Section 1: Setup & Imports

Load API keys from the `.env` file and import all project modules.

In [ ]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Make sure Python can find the src/ folder
sys.path.insert(0, os.path.abspath('.'))

# Load API keys from .env
load_dotenv()

# Import all project modules
from src.sector_mapping import get_tickers_for_sector, list_available_sectors, get_sector_description
from src.market_data import get_market_data, format_market_data_for_prompt
from src.news_loader import load_news_for_tickers
from src.prompt_builder import build_morningnote_prompt
from src.model_runner import run_openai_model, run_anthropic_model, save_note
from src.evaluator import create_model_comparison_template, create_fact_check_template, print_evaluation_instructions

print('✓ All imports successful.')
print(f'  OpenAI key set:    {bool(os.getenv("OPENAI_API_KEY"))}')
print(f'  Anthropic key set: {bool(os.getenv("ANTHROPIC_API_KEY"))}')
print(f'  NewsAPI key set:   {bool(os.getenv("NEWSAPI_KEY"))}')

---
<a id='section-2'></a>
## Section 2: User Selects Sector

The user picks one sector. The system handles everything else automatically.

**Change `SELECTED_SECTOR` below to try a different sector.**

In [ ]:
# Show all available sectors so the user knows their options
print('Available sectors:')
for sector in list_available_sectors():
    desc = get_sector_description(sector)
    print(f'  • {sector}: {desc}')

print()

# ── Change this to try a different sector ────────────────────────────────────
SELECTED_SECTOR = 'AI & Semiconductors'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Selected sector: {SELECTED_SECTOR}')

---
<a id='section-3'></a>
## Section 3: Build Watchlist

The system maps the selected sector to a curated list of tickers automatically.
No manual ticker entry needed.

In [ ]:
tickers = get_tickers_for_sector(SELECTED_SECTOR)

print(f'Sector:  {SELECTED_SECTOR}')
print(f'Tickers: {tickers}')
print(f'Count:   {len(tickers)} companies')

---
<a id='section-4'></a>
## Section 4: Fetch Market Data

Real price data fetched live from Yahoo Finance via `yfinance`.

If data is unavailable (e.g., weekend, invalid ticker), the system raises a
clear error — it does **not** generate fake fallback prices.

In [ ]:
print('Fetching market data from yfinance...')
market_df = get_market_data(tickers)

print(f'✓ Fetched data for {len(market_df)} tickers.\n')
display(market_df)

---
<a id='section-5'></a>
## Section 5: Load Company News

Fetches real news headlines via NewsAPI.
If NewsAPI fails, automatically falls back to `data/sample_news.csv`.

In [ ]:
print('Loading news headlines...')
news_df = load_news_for_tickers(tickers)

print(f'\n✓ Loaded {len(news_df)} news articles.\n')
display(news_df)

---
<a id='section-6'></a>
## Section 6: Build Prompt

Combines the sector name, tickers, market data, and news into a single
structured prompt. The prompt includes anti-hallucination instructions
that tell the model to use cautious language and flag limited evidence.

In [ ]:
prompt = build_morningnote_prompt(
    sector=SELECTED_SECTOR,
    tickers=tickers,
    market_df=market_df,
    news_df=news_df,
)

print(f'✓ Prompt built. Total length: {len(prompt)} characters.\n')
print('--- Prompt Preview (first 800 characters) ---')
print(prompt[:800])
print('...')

---
<a id='section-7'></a>
## Section 7: Generate Morning Notes (Two Models)

Both models receive the **exact same prompt** so the comparison is fair.

- **Model A:** OpenAI `gpt-4o-mini`
- **Model B:** Anthropic `claude-haiku-4-5-20251001`

Outputs are saved to the `outputs/` folder.

In [ ]:
# ── Model A: OpenAI gpt-4o-mini ───────────────────────────────────────────
print('Generating Model A output (OpenAI gpt-4o-mini)...')
model_a_output = run_openai_model(prompt, model_name='gpt-4o-mini')
save_note(model_a_output, 'outputs/model_a_morning_note.md')
print('✓ Model A done.\n')

# ── Model B: Anthropic Claude Haiku ───────────────────────────────────────
print('Generating Model B output (Anthropic claude-haiku-4-5-20251001)...')
model_b_output = run_anthropic_model(prompt, model_name='claude-haiku-4-5-20251001')
save_note(model_b_output, 'outputs/model_b_morning_note.md')
print('✓ Model B done.')

In [ ]:
# Display both outputs side by side for easy reading
print('=' * 60)
print('MODEL A — OpenAI gpt-4o-mini')
print('=' * 60)
display(Markdown(model_a_output))

print('\n')
print('=' * 60)
print('MODEL B — Anthropic Claude Haiku')
print('=' * 60)
display(Markdown(model_b_output))

---
<a id='section-8'></a>
## Section 8: Baseline — No AI

This is what the user would see **without** any AI: just the raw data dumped
as plain text. Comparing this to the AI outputs shows whether the AI adds value.

In [ ]:
from datetime import date

# Build the baseline note — just raw data, no summarization
headlines = '\n'.join(
    f"  - [{row['date']}] {row['ticker']}: {row['headline']} ({row['source']})"
    for _, row in news_df.iterrows()
)

baseline = f"""# Baseline Morning Note (No AI)

Date: {date.today()}
Sector: {SELECTED_SECTOR}
Tickers: {', '.join(tickers)}

## Raw Market Data

{market_df.to_string(index=False)}

## Raw Headlines

{headlines}
"""

# Save to file
with open('outputs/baseline_note.md', 'w', encoding='utf-8') as f:
    f.write(baseline)

print('✓ Baseline saved to outputs/baseline_note.md\n')
print(baseline)

---
<a id='section-9'></a>
## Section 9: Compare Model Outputs

Generate the blank scoring table, then **manually fill it in** after reading
both model outputs above.

Open `outputs/model_comparison.csv` in Excel or a text editor and score
each dimension 1–5 for both models.

In [ ]:
# Generate the blank scoring table
comparison_df = create_model_comparison_template(
    model_a_name='GPT-4o-mini',
    model_b_name='Claude Haiku',
)

print()
display(comparison_df[['dimension', 'description', 'GPT-4o-mini', 'Claude Haiku', 'notes']])

In [ ]:
# After filling in the CSV manually, reload and display the final scores here
# Uncomment this cell once you have filled in the scores:

# filled_df = pd.read_csv('outputs/model_comparison.csv')
# display(filled_df)

---
<a id='section-10'></a>
## Section 10: Fact-Check Claims

Generate the blank fact-check table, then **manually fill it in** by picking
specific sentences from each model output and checking them against the source data.

For each claim, mark: **Yes** (supported) / **Partial** / **No** (hallucination).

In [ ]:
# Generate the blank fact-check table
fact_check_df = create_fact_check_template()

print()
display(fact_check_df)

In [ ]:
# After filling in the CSV manually, reload and display the results here
# Uncomment this cell once you have filled in the fact-check table:

# filled_fc = pd.read_csv('outputs/fact_check_table.csv')
# display(filled_fc)

In [ ]:
# Print step-by-step instructions for the manual evaluation
print_evaluation_instructions()

---
<a id='section-11'></a>
## Section 11: Conclusion and Limitations

**What worked well:**
> Fill in after completing the evaluation — what did the AI do better than the baseline?

**Which model performed better and why:**
> Fill in after scoring both models — refer to your model_comparison.csv.

**Failure case:**
> Fill in after fact-checking — describe one example where a model made an
> unsupported causal claim (e.g., said a stock moved *because of* a headline,
> when the data only shows they happened around the same time).

**Limitations:**
1. Sector-to-ticker mapping is manually curated — not dynamically updated by market cap.
2. NewsAPI free tier limits history to 1 month and 100 requests/day.
3. LLMs may overstate causality even with anti-hallucination instructions.
4. Human review is still required before acting on any AI-generated financial content.
5. This prototype covers 10 sectors; a production tool would need broader coverage.

---

*This notebook is not investment advice. All outputs require human review.*